# SAGE — full-scale training (~500k rows)

End-to-end training run targeting ~500k training examples after dedup — past
the knee of the data/quality curve for a 137M encoder, with headroom for 2–3
iterations per Colab session rather than one all-or-nothing run. Intended for
an A100 80 GB on Colab Pro+; smaller GPUs work with reduced batch size.

Pipeline:
1. Clone repo + install deps
2. Aggregate all 7 real sources at 100k per source (~500k after dedup)
3. (Optional) merge synthetic trajectories already generated on Drive
4. Trajectorize train / val / test
5. Train 3 epochs at seq=2048 with layer-wise LR decay + focal loss + observation masking
6. Export FP32 + INT8 ONNX
7. Smoke test the INT8 artifact

**Before running:**
- Runtime → Change runtime type → **A100 High-RAM** (or L4 — bump `grad_accum`)
- Add a Colab secret named `HF_TOKEN` (key icon in sidebar)
- Mount Drive so a disconnect doesn't throw away compute

Expected wall-clock on A100 80 GB: **~10–14 hrs** for 500k rows × 3 epochs at seq=2048.

## 1. Mount Drive

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/sage'
DATA_DIR = f'{WORK_DIR}/data/processed-1m'
CKPT_DIR = f'{WORK_DIR}/checkpoints/sage-v1-1m'
ART_DIR  = f'{WORK_DIR}/artifacts/sage-v1-1m'
SYN_DIR  = f'{WORK_DIR}/synthetic'  # optional pre-generated synthetic batches
for d in (DATA_DIR, CKPT_DIR, ART_DIR):
    os.makedirs(d, exist_ok=True)
print('work :', WORK_DIR)
print('data :', DATA_DIR)
print('ckpt :', CKPT_DIR)
print('art  :', ART_DIR)

## 2. Clone repo

In [ ]:
%cd /content
!rm -rf sage
!git clone https://github.com/MegalithOfficial/sage.git
%cd sage

## 3. Install

In [ ]:
!pip install -q -e '.[train,export]'

## 4. Auth + GPU sanity

In [ ]:
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

!nvidia-smi
import torch
assert torch.cuda.is_available(), 'no GPU runtime — switch to A100/L4 first'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram_gb:.1f} GB')

## 5. Aggregate ~1M rows from the 7 real sources

Per-source cap is set high enough that the smaller sources saturate naturally while the large ones (Civil Comments, ToxicChat) contribute most volume. Cached HF downloads live on Drive — second run is much faster.

Skip this cell if `DATA_DIR/stats.json` already exists from a prior run.

In [ ]:
import os
os.environ['HF_DATASETS_CACHE'] = f'{WORK_DIR}/hf_cache/datasets'
os.environ['HF_HOME'] = f'{WORK_DIR}/hf_cache'

stats = f'{DATA_DIR}/stats.json'
if os.path.exists(stats):
    print(f'reusing existing aggregate at {DATA_DIR}')
else:
    # 200k per source × 7 sources = 1.4M pre-dedup; ~1M after collision drop.
    !python -m training.data.aggregate \
        --out {DATA_DIR} \
        --sources all \
        --limit-per-source 200000
!cat {DATA_DIR}/stats.json | python -c "import json,sys; s=json.load(sys.stdin); print('splits:'); [print(f\"  {k}: {v['n']}\") for k,v in s['splits'].items()]; print('per-cat (train):'); [print(f'  {k}: {v}') for k,v in s['splits']['train']['per_category_positive'].items()]"

## 6. (Optional) merge pre-generated synthetic trajectories

If you already ran `colab_synthesize.ipynb` for the rare categories (grooming / self_harm / sexual_minors, both polarities) and saved the JSONL batches into `WORK_DIR/synthetic/*.jsonl`, merge them in here. Otherwise skip — the real sources alone still produce ~1M after dedup.

In [ ]:
import os
os.environ['HF_DATASETS_CACHE'] = f'{WORK_DIR}/hf_cache/datasets'
os.environ['HF_HOME'] = f'{WORK_DIR}/hf_cache'

stats = f'{DATA_DIR}/stats.json'
if os.path.exists(stats):
    print(f'reusing existing aggregate at {DATA_DIR}')
else:
    # 100k per source × 7 sources ≈ 600k pre-dedup, ~500k after collision drop.
    # Past ~500k a 137M encoder mostly absorbs label noise; the remaining signal
    # comes from the synthetic trajectories for rare categories merged below.
    !python -m training.data.aggregate \
        --out {DATA_DIR} \
        --sources all \
        --limit-per-source 100000
!cat {DATA_DIR}/stats.json | python -c "import json,sys; s=json.load(sys.stdin); print('splits:'); [print(f\"  {k}: {v['n']}\") for k,v in s['splits'].items()]; print('per-cat (train):'); [print(f'  {k}: {v}') for k,v in s['splits']['train']['per_category_positive'].items()]"

## 7. Trajectorize

In [ ]:
for split in ['train', 'val', 'test']:
    src = f'{DATA_DIR}/{split}.jsonl'
    dst = f'{DATA_DIR}/{split}.traj.jsonl'
    if os.path.exists(dst):
        print(f'skip {split} (already trajectorized)')
        continue
    !python -m training.data.trajectorize_cli --in {src} --out {dst}

# Merge synthetic into the training trajectory file (one-time, idempotent via marker).
import glob
syn_files = sorted(glob.glob(f'{SYN_DIR}/*.jsonl'))
merged_marker = f'{DATA_DIR}/.synthetic_merged'
if syn_files and not os.path.exists(merged_marker):
    train_path = f'{DATA_DIR}/train.traj.jsonl'
    total = 0
    with open(train_path, 'a') as out:
        for p in syn_files:
            with open(p) as f:
                for line in f:
                    line = line.strip()
                    if line:
                        out.write(line + '\n')
                        total += 1
            print(f'  merged {p} (+{total} cumulative)')
    open(merged_marker, 'w').write(f'{total}')
    print(f'merged {total} synthetic rows into {train_path}')

!wc -l {DATA_DIR}/*.traj.jsonl

## 8. Training config

Auto-sized to the detected VRAM. Effective batch is held near 64 regardless of hardware.

In [ ]:
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Jina v2 base materializes the full [batch, heads, seq, seq] attention matrix
# in fp32 — no FlashAttention in its default forward — so seq=2048 is much
# more expensive than the naive VRAM math suggests. Sizes below are measured
# to fit without OOM; don't bump them without testing.
if vram_gb >= 70:      # A100 / H100 80 GB
    batch_size, grad_accum, max_length, num_workers = 6, 8, 2048, 8
elif vram_gb >= 35:    # A100 40 GB
    batch_size, grad_accum, max_length, num_workers = 3, 16, 2048, 6
elif vram_gb >= 22:    # L4 24 GB — attention at 2048 doesn't fit with bs>=2, fall back to 1024
    batch_size, grad_accum, max_length, num_workers = 8, 6, 1024, 4
else:
    raise RuntimeError(f'VRAM {vram_gb:.0f}GB is too small for seq=2048 training')

EPOCHS = 3
LOG_EVERY = 200
SAVE_EVERY_STEPS = 500  # full-state checkpoint frequency — enables mid-epoch resume
print(f'VRAM {vram_gb:.0f} GB → bs={batch_size} accum={grad_accum} seq={max_length} (effective batch {batch_size*grad_accum})')

## 9. Train

Checkpoints stream to Drive (`last.pt` after every epoch, `best.pt` on new best macro-F1). If the runtime drops, re-run **this cell** — it auto-resumes from `last.pt`.

In [ ]:
import os
resume_arg = ''
last_ckpt = f'{CKPT_DIR}/last.pt'
if os.path.exists(last_ckpt):
    resume_arg = f'--resume-from {last_ckpt}'
    print(f'resuming from {last_ckpt}')
else:
    print('fresh training run')

!python -m training.train \
    --train {DATA_DIR}/train.traj.jsonl \
    --val   {DATA_DIR}/val.traj.jsonl \
    --out   {CKPT_DIR} \
    --epochs {EPOCHS} \
    --batch-size {batch_size} \
    --grad-accum {grad_accum} \
    --max-length {max_length} \
    --num-workers {num_workers} \
    --log-every {LOG_EVERY} \
    --save-every-steps {SAVE_EVERY_STEPS} \
    {resume_arg}

## 10. Export ONNX (FP32 + INT8)

In [ ]:
!python -m training.export_onnx \
    --checkpoint {CKPT_DIR}/best.pt \
    --max-length {max_length} \
    --out-fp32 {ART_DIR}/sage-v1-fp32.onnx \
    --out-int8 {ART_DIR}/sage-v1-int8.onnx

!ls -lh {ART_DIR}

## 11. Smoke test the INT8 artifact

In [ ]:
from sage.inference import Sage
model = Sage.from_onnx(f'{ART_DIR}/sage-v1-int8.onnx', max_length=max_length)

tests = [
    'how do I bake sourdough?',
    'I hate all [slur], they ruin everything',
    'what are good study techniques',
    'I want to kill myself',
    'you should meet me alone, don\'t tell your parents',
]
for t in tests:
    r = model.check(t)
    flagged = [c.value for c, v in r.categories.items() if v.flagged]
    top = sorted(r.categories.items(), key=lambda kv: -kv[1].score)[:3]
    top_str = ', '.join(f'{c.value}={v.score:.2f}' for c, v in top)
    print(f'flagged={flagged}  top=[{top_str}]  text={t!r}')

## 12. Benchmark against SAGE-Bench-Hard

In [ ]:
!python -m benchmarks.sage_bench_hard.run \
    --checkpoint {CKPT_DIR}/best.pt \
    --max-length {max_length} \
    --cuda